In [1]:
!pip install torch torchvision torchaudio --quiet

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import unicodedata
import re
import os
import time
import string
import random
from pathlib import Path

print(f"   PyTorch: {torch.__version__}")


   PyTorch: 2.8.0+cu126


In [3]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Seeds set for reproducibility!")


Device: cpu
Seeds set for reproducibility!


In [34]:
print("OPTIMIZED CONFIGURATION")

# Language Configuration
LANGUAGES = [
    'assamese', 'bengali', 'bodo', 'gujarati', 'hindi',
    'kannada', 'kashmiri', 'konkani', 'maithili', 'marathi',
    'manipuri', 'oriya', 'punjabi', 'sanskrit', 'sindhi',
    'tamil', 'telugu', 'urdu'
]

# Model Hyperparameters
HIDDEN_SIZE = 128
EMBEDDING_DIM = 128
NUM_LAYERS = 1
DROPOUT = 0.3

# Training Hyperparameters
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 0.001
WARMUP_STEPS = 1000
GRAD_CLIP = 1.0
TEACHER_FORCING_RATIO = 1.0
LABEL_SMOOTHING = 0.0
WEIGHT_DECAY = 1e-5

#Data Configuration
MAX_PAIRS = 60000
MAX_LENGTH = 30
MIN_LENGTH = 2

# Special tokens
PAD_token = 0
SOS_token = 1
EOS_token = 2

print(f" Model: Hidden={HIDDEN_SIZE}, Embed={EMBEDDING_DIM}, Layers={NUM_LAYERS}")
print(f" Training: Batch={BATCH_SIZE}, LR={LEARNING_RATE}, Epochs={EPOCHS}")
print(f" Data: Max={MAX_PAIRS:,} pairs, Length={MIN_LENGTH}-{MAX_LENGTH}")
print(f" Optimizations: Warmup={WARMUP_STEPS}, Smoothing={LABEL_SMOOTHING}")


OPTIMIZED CONFIGURATION
 Model: Hidden=128, Embed=128, Layers=1
 Training: Batch=64, LR=0.001, Epochs=30
 Data: Max=60,000 pairs, Length=2-30
 Optimizations: Warmup=1000, Smoothing=0.0


In [53]:
# Quick test with one language
LANGUAGES = ['telugu']

In [7]:
class CharVocab:
    """Character-level vocabulary with frequency tracking"""

    def __init__(self, name):
        self.name = name
        self.char2index = {'<PAD>': PAD_token, '<SOS>': SOS_token, '<EOS>': EOS_token}
        self.char2count = {}
        self.index2char = {PAD_token: '<PAD>', SOS_token: '<SOS>', EOS_token: '<EOS>'}
        self.n_chars = 3

    def addString(self, string):
        """Add all characters from string"""
        for char in string:
            self.addChar(char)

    def addChar(self, char):
        """Add single character and update frequency"""
        if char not in self.char2index:
            self.char2index[char] = self.n_chars
            self.char2count[char] = 1
            self.index2char[self.n_chars] = char
            self.n_chars += 1
        else:
            self.char2count[char] += 1

print(" CharVocab class defined!")


 CharVocab class defined!


In [8]:
def clean_romanized(text):
    """
    Enhanced cleaning for romanized text
    - Lowercase, remove punctuation, digits
    - NFKC normalization (CRITICAL)
    """
    if pd.isnull(text) or not text:
        return ''

    text = str(text).strip().lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = unicodedata.normalize('NFKC', text)  # CRITICAL
    text = ' '.join(text.split())
    return text


def clean_indic(text):
    """
    Enhanced cleaning for Indic scripts
    - NFKC normalization (CRITICAL for Indic)
    """
    if pd.isnull(text) or not text:
        return ''

    text = str(text).strip()
    text = unicodedata.normalize('NFKC', text)  # CRITICAL
    text = ' '.join(text.split())
    return text


def validate_pair(src, tgt):
    """Validate transliteration pair"""
    return (src and tgt and
            MIN_LENGTH <= len(src) <= MAX_LENGTH and
            MIN_LENGTH <= len(tgt) <= MAX_LENGTH)

print(" Enhanced cleaning functions defined!")


 Enhanced cleaning functions defined!


In [9]:
def load_language_data(language, base_path, max_pairs=MAX_PAIRS):
    """
    Load with STRATIFIED SAMPLING by length
    Ensures diverse length distribution
    """

    # Try multiple file naming patterns
    possible_files = [
        f"{language[:3]}_train.csv"
    ]

    df = None
    for filename in possible_files:
        filepath = os.path.join(base_path, filename)
        if os.path.exists(filepath):
            print(f" Loading: {filename}")
            try:
                df = pd.read_csv(filepath, header=None,
                               names=['source', 'target'],
                               encoding='utf-8')
                break
            except Exception as e:
                print(f"Error: {e}")

    if df is None:
        print(f"No data found for {language}")
        return None

    # Data cleaning pipeline
    initial_count = len(df)
    df = df.dropna().copy()

    # Apply enhanced cleaning
    df['source'] = df['source'].apply(clean_romanized)
    df['target'] = df['target'].apply(clean_indic)

    # Remove empty strings
    df = df[(df['source'] != '') & (df['target'] != '')]

    # Remove duplicates
    df = df.drop_duplicates()

    # Validate pairs
    df = df[df.apply(lambda x: validate_pair(x.source, x.target), axis=1)]

    # STRATIFIED SAMPLING if too many pairs
    if len(df) > max_pairs:
        print(f"Stratified sampling: {len(df)} → {max_pairs}")

        df['src_len'] = df['source'].str.len()

        # Sample from length bins
        sampled_dfs = []
        for length_bin in range(MIN_LENGTH, MAX_LENGTH + 1, 3):
            bin_df = df[(df['src_len'] >= length_bin) &
                       (df['src_len'] < length_bin + 3)]

            if len(bin_df) > 0:
                sample_size = min(len(bin_df), max_pairs // 10)
                sampled_dfs.append(bin_df.sample(n=sample_size, random_state=42))

        df = pd.concat(sampled_dfs, ignore_index=True)
        df = df.drop(['src_len'], axis=1)
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"  {language}: {initial_count} → {len(df)} pairs")
    return df

print("Smart data loading defined!")


Smart data loading defined!


In [10]:
class TransliterationDataset(Dataset):
    """Memory-efficient transliteration dataset"""

    def __init__(self, pairs, src_vocab, tgt_vocab, max_length):
        self.pairs = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]

        # Convert to indices
        src_indices = ([SOS_token] +
                      [self.src_vocab.char2index.get(c, PAD_token) for c in src] +
                      [EOS_token])
        tgt_indices = ([SOS_token] +
                      [self.tgt_vocab.char2index.get(c, PAD_token) for c in tgt] +
                      [EOS_token])

        # Pad sequences
        src_padded = self._pad(src_indices)
        tgt_padded = self._pad(tgt_indices)

        return (torch.tensor(src_padded, dtype=torch.long),
                torch.tensor(tgt_padded, dtype=torch.long))

    def _pad(self, seq):
        """Pad or truncate to max_length"""
        if len(seq) >= self.max_length:
            return seq[:self.max_length]
        return seq + [PAD_token] * (self.max_length - len(seq))

print("TransliterationDataset defined!")

TransliterationDataset defined!


In [11]:
class EncoderRNN(nn.Module):
    """
    Optimized GRU Encoder
    - Separate embedding (128D)
    - Single layer (efficiency)
    - Xavier initialization
    """

    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.3):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_size, EMBEDDING_DIM,
                                     padding_idx=PAD_token)
        self.gru = nn.GRU(EMBEDDING_DIM, hidden_size, num_layers,
                         batch_first=True,
                         dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        """Xavier initialization for stability"""
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

    def forward(self, input_seq):
        embedded = self.dropout(self.embedding(input_seq))
        outputs, hidden = self.gru(embedded)
        return outputs, hidden

print("Optimized EncoderRNN defined!")

Optimized EncoderRNN defined!


In [12]:
class BahdanauAttention(nn.Module):
    """
    Fixed Bahdanau (additive) attention for correct tensor broadcasting.
    """
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Ua = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Va = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, query, keys):
        """
        query: (batch, hidden) or (num_layers, batch, hidden)
        keys: (batch, seq_len, hidden)
        """
        # If query has extra leading dims (e.g., [1, batch, hidden]), flatten it
        if query.dim() == 3:
            # Use the last layer's hidden state
            query = query[-1]
        # Now query is (batch, hidden)
        batch_size, seq_len, hidden = keys.size()
        query = query.unsqueeze(1).expand(-1, seq_len, -1)  # (batch, seq_len, hidden)
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))  # (batch, seq_len, 1)
        scores = scores.squeeze(2)
        weights = F.softmax(scores, dim=1).unsqueeze(1)
        context = torch.bmm(weights, keys)
        return context, weights

In [13]:
class AttnDecoderRNN(nn.Module):
    """
    Optimized GRU Decoder with Attention
    - Separate embedding
    - Single layer
    - Xavier initialization
    """

    def __init__(self, hidden_size, output_size, num_layers=1, dropout=0.3):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(output_size, EMBEDDING_DIM,
                                     padding_idx=PAD_token)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(EMBEDDING_DIM + hidden_size, hidden_size, num_layers,
                         batch_first=True,
                         dropout=dropout if num_layers > 1 else 0)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        """Xavier initialization"""
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.full((batch_size, 1), SOS_token,
                                  dtype=torch.long, device=device)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)

            if target_tensor is not None and random.random() < TEACHER_FORCING_RATIO:
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        return decoder_outputs

    def forward_step(self, input, hidden, encoder_outputs):
        """Single decoding step with attention"""
        embedded = self.dropout(self.embedding(input))

        query = hidden[-1] if self.num_layers > 1 else hidden
        context, _ = self.attention(query, encoder_outputs)

        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.gru(rnn_input, hidden)
        output = self.out(output)

        return output, hidden

print("Optimized AttnDecoderRNN defined!")

Optimized AttnDecoderRNN defined!


In [14]:
class WarmupScheduler:
    """
    Learning rate warmup scheduler
    Gradually increases LR from 0 to target
    """

    def __init__(self, optimizer, warmup_steps, initial_lr):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.initial_lr = initial_lr
        self.step_num = 0

    def step(self):
        """Update learning rate"""
        self.step_num += 1
        if self.step_num <= self.warmup_steps:
            lr = self.initial_lr * (self.step_num / self.warmup_steps)
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr

    def get_lr(self):
        """Get current LR"""
        if self.step_num <= self.warmup_steps:
            return self.initial_lr * (self.step_num / self.warmup_steps)
        return self.initial_lr

print("WarmupScheduler defined!")

WarmupScheduler defined!


In [15]:
def label_smoothing_loss(outputs, targets, smoothing=0.1, ignore_index=0):
    """
    Manual label smoothing implementation for PyTorch compatibility
    Works on ALL PyTorch versions
    """
    vocab_size = outputs.size(-1)

    # Create smoothed labels
    confidence = 1.0 - smoothing
    smooth_value = smoothing / (vocab_size - 1)

    # One-hot encoding
    one_hot = torch.zeros_like(outputs).scatter(1, targets.unsqueeze(1), 1)

    # Apply smoothing
    smooth_one_hot = one_hot * confidence + (1 - one_hot) * smooth_value

    # Calculate loss
    loss = -(smooth_one_hot * outputs).sum(dim=-1)

    # Mask padding
    mask = (targets != ignore_index).float()
    loss = (loss * mask).sum() / mask.sum()

    return loss

In [16]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
               decoder_optimizer, warmup_scheduler=None):
    """
    Train one epoch with all optimizations:
    - Manual label smoothing
    - Gradient clipping
    - Warmup scheduling
    - NaN detection
    """

    encoder.train()
    decoder.train()

    total_loss = 0
    valid_batches = 0

    for batch_idx, (input_tensor, target_tensor) in enumerate(dataloader):
        input_tensor = input_tensor.to(device)
        target_tensor = target_tensor.to(device)

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        # Forward pass
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs = decoder(encoder_outputs, encoder_hidden, target_tensor)

        # Manual label smoothing loss
        loss = label_smoothing_loss(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1),
            smoothing=LABEL_SMOOTHING,
            ignore_index=PAD_token
        )

        # NaN detection
        if torch.isnan(loss):
            print(f"NaN at batch {batch_idx}")
            continue

        # Backward pass
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), GRAD_CLIP)
        torch.nn.utils.clip_grad_norm_(decoder.parameters(), GRAD_CLIP)

        encoder_optimizer.step()
        decoder_optimizer.step()

        # Warmup step
        if warmup_scheduler:
            warmup_scheduler.step()

        total_loss += loss.item()
        valid_batches += 1

    return total_loss / max(valid_batches, 1)

print("Training function with manual label smoothing defined!")

Training function with manual label smoothing defined!


In [17]:
def evaluate_model(encoder, decoder, sentence, src_vocab, tgt_vocab):
    """Evaluate single sentence (inference)"""

    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        # Prepare input
        cleaned = clean_romanized(sentence)
        input_indices = ([SOS_token] +
                        [src_vocab.char2index.get(c, PAD_token) for c in cleaned] +
                        [EOS_token])

        # Pad
        if len(input_indices) >= MAX_LENGTH:
            input_indices = input_indices[:MAX_LENGTH]
        else:
            input_indices += [PAD_token] * (MAX_LENGTH - len(input_indices))

        input_tensor = torch.tensor([input_indices], dtype=torch.long, device=device)

        # Encode
        encoder_outputs, encoder_hidden = encoder(input_tensor)

        # Decode
        decoder_outputs = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_chars = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                break
            if idx.item() > 2:
                decoded_chars.append(tgt_vocab.index2char.get(idx.item(), ''))

        return ''.join(decoded_chars)

print("Evaluation function defined!")

Evaluation function defined!


In [18]:
def train_language_model(language, train_pairs, src_vocab, tgt_vocab, epochs=EPOCHS):
    """
    Complete training pipeline with ALL optimizations:
    - AdamW, Warmup, Manual label smoothing
    - ReduceLROnPlateau, Early stopping
    """


    print(f"TRAINING: {language.upper()}")
    print(f"Pairs: {len(train_pairs):,}")
    print(f"Vocab: src={src_vocab.n_chars}, tgt={tgt_vocab.n_chars}")

    # Dataset & DataLoader
    dataset = TransliterationDataset(train_pairs, src_vocab, tgt_vocab, MAX_LENGTH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                           shuffle=True, drop_last=True, num_workers=0)

    print(f"Batches: {len(dataloader)}")

    # Models
    encoder = EncoderRNN(src_vocab.n_chars, HIDDEN_SIZE,
                        NUM_LAYERS, DROPOUT).to(device)
    decoder = AttnDecoderRNN(HIDDEN_SIZE, tgt_vocab.n_chars,
                            NUM_LAYERS, DROPOUT).to(device)

    # AdamW optimizers with weight decay
    encoder_optimizer = optim.AdamW(encoder.parameters(), lr=LEARNING_RATE,
                                   weight_decay=WEIGHT_DECAY)
    decoder_optimizer = optim.AdamW(decoder.parameters(), lr=LEARNING_RATE,
                                   weight_decay=WEIGHT_DECAY)

    # Warmup scheduler
    warmup_scheduler = WarmupScheduler(encoder_optimizer, WARMUP_STEPS, LEARNING_RATE)

    # ReduceLROnPlateau (NO verbose for old PyTorch compatibility)
    encoder_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        encoder_optimizer, mode='min', factor=0.5, patience=2
    )
    decoder_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        decoder_optimizer, mode='min', factor=0.5, patience=2
    )

    # Early stopping
    best_loss = float('inf')
    patience_counter = 0
    max_patience = 5

    print(f"\nTraining...")
    print(f"Manual label smoothing: {LABEL_SMOOTHING}")

    # Training loop
    for epoch in range(1, epochs + 1):
        start_time = time.time()

        # Train with manual label smoothing
        loss = train_epoch(dataloader, encoder, decoder,
                          encoder_optimizer, decoder_optimizer,
                          warmup_scheduler if epoch == 1 else None)

        epoch_time = time.time() - start_time
        current_lr = encoder_optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch:02}/{epochs} | Loss: {loss:.4f} | " +
              f"LR: {current_lr:.6f} | Time: {epoch_time:.1f}s")

        # Update schedulers
        encoder_scheduler.step(loss)
        decoder_scheduler.step(loss)

        # Early stopping
        if loss < best_loss:
            best_loss = loss
            patience_counter = 0

            torch.save({
                'encoder': encoder.state_dict(),
                'decoder': decoder.state_dict(),
                'src_vocab': src_vocab,
                'tgt_vocab': tgt_vocab,
                'config': {
                    'hidden_size': HIDDEN_SIZE,
                    'embedding_dim': EMBEDDING_DIM,
                    'num_layers': NUM_LAYERS,
                    'dropout': DROPOUT,
                    'max_length': MAX_LENGTH
                }
            }, f'best_{language}_model.pth')

            print(f"Best model saved (loss: {loss:.4f})")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{max_patience}")

            if patience_counter >= max_patience:
                print(f"Early stopping!")
                break

    print(f"\nComplete! Best: {best_loss:.4f}")
    return encoder, decoder

print("Training pipeline with manual label smoothing defined!")

Training pipeline with manual label smoothing defined!


In [41]:
def test_random_samples(language, encoder, decoder, test_pairs,
                       src_vocab, tgt_vocab, num_samples=10):
    """Test on random samples"""

    # print(f"\n{'='*80}")
    print(f"TESTING: {language.upper()}")
    # print(f"{'='*80}")

    samples = random.sample(test_pairs, min(num_samples, len(test_pairs)))
    correct = 0

    for idx, (src, tgt) in enumerate(samples, 1):
        pred = evaluate_model(encoder, decoder, src, src_vocab, tgt_vocab)
        match = " " if pred == tgt else " "
        if pred == tgt:
            correct += 1

        print(f"\n  [{idx}]")
        print(f"      Input:     {src}")
        print(f"      Target:    {tgt}")
        print(f"      Predicted: {pred} {match}")

    accuracy = (correct / len(samples)) * 100
    print(f"\nAccuracy: {accuracy:.1f}% ({correct}/{len(samples)})")
    #print(f"{'='*80}\n")

print("Testing function defined!")

Testing function defined!


In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [54]:
print("DATA LOADING")


BASE_DATA_PATH = '/content/drive/MyDrive/aksharantar_sampled'

# Storage
all_language_data = {}
all_src_vocabs = {}
all_tgt_vocabs = {}

print(f"\nLoading {len(LANGUAGES)} languages...")

# Load each language
for lang in LANGUAGES:
    lang_folder = os.path.join(BASE_DATA_PATH, lang[:3])

    df = load_language_data(lang, lang_folder, MAX_PAIRS)

    if df is not None and len(df) > 0:
        all_language_data[lang] = df

        # Build vocabularies
        src_vocab = CharVocab(f'{lang}_source')
        tgt_vocab = CharVocab(f'{lang}_target')

        for _, row in df.iterrows():
            src_vocab.addString(row['source'])
            tgt_vocab.addString(row['target'])

        all_src_vocabs[lang] = src_vocab
        all_tgt_vocabs[lang] = tgt_vocab

        print(f"{lang}: src={src_vocab.n_chars}, tgt={tgt_vocab.n_chars}")

print(f"\nLoaded {len(all_language_data)} languages!")

DATA LOADING

Loading 1 languages...
 Loading: tel_train.csv
  telugu: 51200 → 51199 pairs
telugu: src=29, tgt=65

Loaded 1 languages!


In [55]:
print("TRAINING ALL MODELS")

trained_models = {}

for lang in all_language_data.keys():
    df = all_language_data[lang]
    src_vocab = all_src_vocabs[lang]
    tgt_vocab = all_tgt_vocabs[lang]

    # 80/20 split
    split_idx = int(len(df) * 0.8)
    train_df = df[:split_idx]
    test_df = df[split_idx:]

    # Create pairs
    train_pairs = [(row.source, row.target)
                   for row in train_df.itertuples(index=False)]
    test_pairs = [(row.source, row.target)
                  for row in test_df.itertuples(index=False)]

    # Train
    encoder, decoder = train_language_model(
        lang, train_pairs, src_vocab, tgt_vocab, EPOCHS
    )

    if encoder is not None:
        trained_models[lang] = (encoder, decoder, test_pairs)

    print()

print(f"TRAINED {len(trained_models)} MODELS!")

TRAINING ALL MODELS
TRAINING: TELUGU
Pairs: 40,959
Vocab: src=29, tgt=65
Batches: 639

Training...
Manual label smoothing: 0.0
Epoch 01/30 | Loss: 2.0454 | LR: 0.000639 | Time: 300.9s
Best model saved (loss: 2.0454)
Epoch 02/30 | Loss: 0.4710 | LR: 0.000639 | Time: 302.6s
Best model saved (loss: 0.4710)
Epoch 03/30 | Loss: 0.2844 | LR: 0.000639 | Time: 302.6s
Best model saved (loss: 0.2844)
Epoch 04/30 | Loss: 0.2331 | LR: 0.000639 | Time: 303.5s
Best model saved (loss: 0.2331)
Epoch 05/30 | Loss: 0.2104 | LR: 0.000639 | Time: 303.8s
Best model saved (loss: 0.2104)
Epoch 06/30 | Loss: 0.1902 | LR: 0.000639 | Time: 300.7s
Best model saved (loss: 0.1902)
Epoch 07/30 | Loss: 0.1772 | LR: 0.000639 | Time: 301.1s
Best model saved (loss: 0.1772)
Epoch 08/30 | Loss: 0.1694 | LR: 0.000639 | Time: 299.5s
Best model saved (loss: 0.1694)
Epoch 09/30 | Loss: 0.1575 | LR: 0.000639 | Time: 297.7s
Best model saved (loss: 0.1575)
Epoch 10/30 | Loss: 0.1517 | LR: 0.000639 | Time: 298.4s
Best model save

In [56]:
print("TESTING ALL MODELS")

for lang in trained_models.keys():
    encoder, decoder, test_pairs = trained_models[lang]
    src_vocab = all_src_vocabs[lang]
    tgt_vocab = all_tgt_vocabs[lang]

    test_random_samples(lang, encoder, decoder, test_pairs,
                       src_vocab, tgt_vocab, num_samples=10)


TESTING ALL MODELS
TESTING: TELUGU

  [1]
      Input:     nelaloki
      Target:    నేలలోకి
      Predicted: నెలలోకి  

  [2]
      Input:     aamevalla
      Target:    ఆమెవల్ల
      Predicted: ఆమెవల్ల  

  [3]
      Input:     issues
      Target:    ఇష్యూస్
      Predicted: ఐస్స్యూస్  

  [4]
      Input:     mudivestundi
      Target:    ముడివేస్తుంది
      Predicted: ముడివేస్తుంది  

  [5]
      Input:     nirvahanhaloenuu
      Target:    నిర్వహణలోనూ
      Predicted: నిర్వహణాలోనూ  

  [6]
      Input:     santarinchukunenduku
      Target:    సంతరించుకునేందుకు
      Predicted: సంతరించుకునేందుకు  

  [7]
      Input:     kalchaledani
      Target:    కాల్చలేదని
      Predicted: కల్చలేదని  

  [8]
      Input:     seppadu
      Target:    సెప్పడు
      Predicted: సెప్పాడు  

  [9]
      Input:     palakagalame
      Target:    పలకగలమే
      Predicted: పాలకగలమే  

  [10]
      Input:     maharashtraku
      Target:    మహారాష్ట్రకు
      Predicted: మహారాష్ట్రకు  

Accuracy: 40.0% (4

In [57]:
print("SAVING ALL MODELS")

for lang in trained_models.keys():
    encoder, decoder, _ = trained_models[lang]
    src_vocab = all_src_vocabs[lang]
    tgt_vocab = all_tgt_vocabs[lang]

    torch.save({
        'encoder_state': encoder.state_dict(),
        'decoder_state': decoder.state_dict(),
        'src_vocab': src_vocab,
        'tgt_vocab': tgt_vocab,
        'config': {
            'hidden_size': HIDDEN_SIZE,
            'embedding_dim': EMBEDDING_DIM,
            'num_layers': NUM_LAYERS,
            'dropout': DROPOUT,
            'max_length': MAX_LENGTH
        }
    }, f'{lang}_transliteration_final.pth')

    print(f"Saved: {lang}_transliteration_final.pth")

print("\nALL MODELS SAVED!")

SAVING ALL MODELS
Saved: telugu_transliteration_final.pth

ALL MODELS SAVED!


In [58]:
def load_and_transliterate(language, word):
    """Load model and transliterate word"""

    try:
        checkpoint = torch.load(f'{language}_transliteration_final.pth')

        src_vocab = checkpoint['src_vocab']
        tgt_vocab = checkpoint['tgt_vocab']
        config = checkpoint['config']

        encoder = EncoderRNN(src_vocab.n_chars, config['hidden_size'],
                            config['num_layers'], config['dropout']).to(device)
        decoder = AttnDecoderRNN(config['hidden_size'], tgt_vocab.n_chars,
                                config['num_layers'], config['dropout']).to(device)

        encoder.load_state_dict(checkpoint['encoder_state'])
        decoder.load_state_dict(checkpoint['decoder_state'])

        result = evaluate_model(encoder, decoder, word, src_vocab, tgt_vocab)
        return result

    except Exception as e:
        return f"Error: {str(e)}"

print("Inference function defined!")

Inference function defined!


In [60]:
print("EXAMPLE TRANSLITERATIONS")

examples = {
    'hindi': ['namaste', 'dhanyavaad', 'bharat'],
    'bengali': ['nomoshkar', 'dhonnobad'],
    'tamil': ['vanakkam', 'nandri']
}

for lang, words in examples.items():
    if lang in trained_models:
        print(f"\n{lang.upper()}:")
        for word in words:
            result = load_and_transliterate(lang, word)
            print(f"  {word:15s} → {result}")

print("PIPELINE COMPLETE!")

EXAMPLE TRANSLITERATIONS
PIPELINE COMPLETE!
